# 06. Simple LangGraph RAG + Web Search

Minimal RAG with a **LangGraph** harness:

1. Retrieve from a tiny in-memory nutrition KB
2. If KB is relevant → answer from KB
3. If KB miss but question is nutrition-related → **web search** → answer from snippets
4. Otherwise → refuse

Then chat through the same NutriBot UI (`static/index.html`).

**Requires:** `GROQ_API_KEY` in `.env` (repo root).

## 1. Setup

In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in .env before running this notebook"
print("Project root:", ROOT)

In [ ]:
# Optional: install notebook-only deps if missing
import importlib.util
import subprocess

missing = [pkg for pkg in ("langgraph", "ddgs") if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call(["uv", "pip", "install", *missing, "--python", sys.executable])
    print("Installed:", missing)
else:
    print("langgraph + ddgs already available")


## 2. Tiny in-memory knowledge base + TF-IDF retriever

No Pinecone — just a few nutrition docs and cosine similarity over TF-IDF vectors.

In [ ]:
from dataclasses import dataclass
from typing import List, Optional, TypedDict, Literal

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

KB_DOCS = [
    "Protein is essential for muscle growth and repair. The recommended daily intake is 0.8g per kilogram of body weight for sedentary adults, but athletes may need 1.2-2.0g/kg.",
    "Carbohydrates are the body's primary energy source. Complex carbs like whole grains provide sustained energy, while simple carbs provide quick energy but can cause blood sugar spikes.",
    "Healthy fats from sources like avocados, nuts, and olive oil are important for hormone production and nutrient absorption. Aim for 20-35% of daily calories from fat.",
    "Vitamins and minerals are essential micronutrients. Vitamin D supports bone health, B vitamins support energy metabolism, and calcium is crucial for bone strength.",
    "Water intake recommendations vary, but the general guideline is 8 glasses per day. However, needs depend on activity level, climate, and individual factors. Drink enough to keep urine pale yellow.",
    "Fiber is important for digestive health. Aim for 25-30g daily from sources like whole grains, legumes, vegetables, and fruits. Fiber helps with satiety and blood sugar control.",
    "Weight loss requires a calorie deficit. A deficit of 500 calories per day typically results in about 0.5kg (1 lb) weight loss per week.",
    "Sodium intake should be limited to less than 2,300mg per day. Excess sodium can increase blood pressure. Choose fresh foods over processed ones when possible.",
]

@dataclass
class Hit:
    content: str
    score: float
    source: str = "kb"
    url: Optional[str] = None
    title: Optional[str] = None


class SimpleKB:
    """TF-IDF retriever over a fixed document list."""

    def __init__(self, docs: List[str], min_score: float = 0.12):
        self.docs = docs
        self.min_score = min_score
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.matrix = self.vectorizer.fit_transform(docs)

    def retrieve(self, query: str, top_k: int = 3) -> List[Hit]:
        q = self.vectorizer.transform([query])
        scores = cosine_similarity(q, self.matrix)[0]
        idxs = np.argsort(scores)[::-1][:top_k]
        hits = [Hit(content=self.docs[i], score=float(scores[i])) for i in idxs if scores[i] >= self.min_score]
        return hits


kb = SimpleKB(KB_DOCS)
print("KB size:", len(KB_DOCS))
print("Sample retrieve:", kb.retrieve("How much protein do I need?")[:2])

## 3. Web search tool + LLM

DuckDuckGo via `ddgs` (no API key). Groq for generation.

In [ ]:
from ddgs import DDGS
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

NUTRITION_KEYWORDS = {
    "protein", "carb", "carbohydrate", "fat", "vitamin", "mineral", "fiber",
    "calorie", "diet", "dietary", "nutrition", "nutrient", "food", "meal", "water",
    "sodium", "sugar", "iron", "calcium", "weight", "muscle", "supplement",
    "omega", "cholesterol", "glucose", "macro", "micro", "vegan", "keto",
    "usda", "guideline", "guidelines", "intake", "serving",
}


def is_nutrition_related(question: str) -> bool:
    tokens = set(question.lower().replace("?", " ").split())
    return bool(tokens & NUTRITION_KEYWORDS)


def web_search(query: str, max_results: int = 5) -> List[Hit]:
    """Search the web and return snippet hits (DuckDuckGo via ddgs)."""
    results: List[Hit] = []
    for i, r in enumerate(DDGS().text(query, max_results=max_results)):
        results.append(
            Hit(
                content=r.get("body") or "",
                score=1.0 - i * 0.05,
                source="web",
                url=r.get("href") or r.get("url"),
                title=r.get("title"),
            )
        )
    return results


KB_SYSTEM = """You are NutriBot. Answer ONLY from the provided knowledge-base context.
Cite sources as [Source 1], [Source 2]. If the context is insufficient, say you don't know.
This is general nutrition information, not medical advice."""

WEB_SYSTEM = """You are NutriBot answering from web search snippets.
Rules:
- Use only the provided snippets; do not invent facts.
- Cite sources as [Source 1], [Source 2] next to claims.
- Prefer reputable health/nutrition sources when snippets conflict.
- Note when evidence looks thin.
- This is general information, not personalized medical advice."""

REFUSAL = (
    "I'm sorry, that's outside what I can help with. I can only answer nutrition "
    "questions. Try asking about foods, nutrients, vitamins, or diet."
)


def format_context(hits: List[Hit]) -> str:
    parts = []
    for i, h in enumerate(hits, 1):
        label = h.title or f"Source {i}"
        url = f" ({h.url})" if h.url else ""
        parts.append(f"[Source {i}] {label}{url}\n{h.content}")
    return "\n\n".join(parts)


def generate(question: str, hits: List[Hit], system: str) -> str:
    context = format_context(hits)
    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),
    ]
    return llm.invoke(messages).content


print("Nutrition check:", is_nutrition_related("How much protein daily?"))
print("Off-topic check:", is_nutrition_related("Who won the World Cup?"))


## 4. LangGraph harness

```
retrieve_kb → route → generate_kb
                   → web_search → generate_web
                   → refuse
```

In [ ]:
from langgraph.graph import StateGraph, START, END


class GraphState(TypedDict):
    question: str
    kb_hits: List[Hit]
    web_hits: List[Hit]
    route: Literal["kb", "web", "refused"]
    answer: str


def node_retrieve_kb(state: GraphState) -> dict:
    hits = kb.retrieve(state["question"], top_k=3)
    return {"kb_hits": hits}


def node_route(state: GraphState) -> dict:
    if state["kb_hits"]:
        return {"route": "kb"}
    if is_nutrition_related(state["question"]):
        return {"route": "web"}
    return {"route": "refused"}


def pick_path(state: GraphState) -> str:
    return state["route"]


def node_generate_kb(state: GraphState) -> dict:
    answer = generate(state["question"], state["kb_hits"], KB_SYSTEM)
    return {"answer": answer}


def node_web_search(state: GraphState) -> dict:
    hits = web_search(state["question"], max_results=5)
    return {"web_hits": hits}


def node_generate_web(state: GraphState) -> dict:
    hits = state.get("web_hits") or []
    if not hits:
        return {"answer": REFUSAL, "route": "refused"}
    answer = generate(state["question"], hits, WEB_SYSTEM)
    return {"answer": answer}


def node_refuse(state: GraphState) -> dict:
    return {"answer": REFUSAL}


graph = StateGraph(GraphState)
graph.add_node("retrieve_kb", node_retrieve_kb)
graph.add_node("route", node_route)
graph.add_node("generate_kb", node_generate_kb)
graph.add_node("web_search", node_web_search)
graph.add_node("generate_web", node_generate_web)
graph.add_node("refuse", node_refuse)

graph.add_edge(START, "retrieve_kb")
graph.add_edge("retrieve_kb", "route")
graph.add_conditional_edges(
    "route",
    pick_path,
    {"kb": "generate_kb", "web": "web_search", "refused": "refuse"},
)
graph.add_edge("generate_kb", END)
graph.add_edge("web_search", "generate_web")
graph.add_edge("generate_web", END)
graph.add_edge("refuse", END)

app_graph = graph.compile()
print("LangGraph compiled:", app_graph.get_graph().nodes.keys())

In [ ]:
import time


def ask(question: str) -> dict:
    """Run the LangGraph RAG harness and return a UI-friendly payload."""
    t0 = time.time()
    result = app_graph.invoke(
        {
            "question": question,
            "kb_hits": [],
            "web_hits": [],
            "route": "refused",
            "answer": "",
        }
    )
    route = result["route"]
    hits = result["kb_hits"] if route == "kb" else result.get("web_hits") or []

    sources = []
    scores = []
    for h in hits:
        if h.url:
            sources.append(f"{h.title or 'Web'} — {h.url}\n{h.content}")
        else:
            sources.append(h.content)
        scores.append(h.score)

    return {
        "question": question,
        "answer": result["answer"],
        "sources": sources,
        "scores": scores,
        "metrics": {
            "total_latency_s": time.time() - t0,
            "route": route,
        },
    }


# Smoke tests: KB hit, web fallback, refuse
for q in [
    "How much protein do I need daily?",
    "What are the latest USDA dietary guidelines for added sugar?",
    "Who won the World Cup?",
]:
    out = ask(q)
    print("=" * 60)
    print("Q:", q)
    print("route:", out["metrics"]["route"], f"({out['metrics']['total_latency_s']:.2f}s)")
    print("A:", out["answer"][:400], "..." if len(out["answer"]) > 400 else "")
    print("sources:", len(out["sources"]))

## 5. Serve the NutriBot UI

Starts a tiny FastAPI app that reuses `static/index.html` and wires `/chat` to the LangGraph harness.

In [ ]:
import threading
from typing import Optional

import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field

PORT = 8765
STATIC_DIR = ROOT / "static"

ui_app = FastAPI(title="NutriBot LangGraph Demo")
ui_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)
ui_app.mount("/static", StaticFiles(directory=STATIC_DIR), name="static")


class ChatRequest(BaseModel):
    question: str
    prompt_type: str = "standard"
    top_k: Optional[int] = None


class ChatResponse(BaseModel):
    question: str
    answer: str
    sources: list[str]
    scores: list[float]
    metrics: dict


@ui_app.get("/")
async def root():
    return FileResponse(STATIC_DIR / "index.html")


@ui_app.get("/health")
async def health():
    return {
        "status": "healthy",
        "uptime_seconds": 0,
        "retriever_type": "SimpleKB+LangGraph",
        "embedder": "tfidf",
    }


@ui_app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    if not request.question.strip():
        raise HTTPException(status_code=400, detail="Question cannot be empty")
    try:
        result = ask(request.question)
        # Surface route in the answer meta line the UI already shows via metrics
        route = result["metrics"].get("route", "?")
        result["answer"] = f"*{'(via ' + route + ')'}*\n\n" + result["answer"]
        return ChatResponse(**result)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


def _run():
    uvicorn.run(ui_app, host="127.0.0.1", port=PORT, log_level="warning")


if not globals().get("_ui_thread_started"):
    t = threading.Thread(target=_run, daemon=True)
    t.start()
    _ui_thread_started = True
    print(f"UI running at http://127.0.0.1:{PORT}")
else:
    print(f"UI already running at http://127.0.0.1:{PORT}")

In [ ]:
from IPython.display import IFrame, display, HTML

display(HTML(f'<p>Open the chat UI: <a href="http://127.0.0.1:{PORT}" target="_blank">http://127.0.0.1:{PORT}</a></p>'))
display(IFrame(src=f"http://127.0.0.1:{PORT}", width="100%", height=720))

## Try these in the UI

| Question | Expected route |
|---|---|
| How much protein do I need daily? | `kb` |
| What are the latest USDA guidelines on added sugar? | `web` (not in tiny KB) |
| Who won the World Cup? | `refused` |

The answer prefix `*(via kb|web|refused)*` shows which LangGraph path ran.